In [12]:
!pip install ultralytics -q

In [13]:
import shutil
from ultralytics import YOLO

shutil.copy('/kaggle/input/datasets/ashtonmears/yolo11n-v1/yolo11n.pt',
            '/kaggle/working/yolo11n.pt')

model = YOLO('/kaggle/working/yolo11n.pt')
print('Model loaded')
print('Classes:', model.names)

Model loaded
Classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'ce

In [14]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        if d.lower() == 'forks':
            print('Found Forks at:', os.path.join(root, d))
            print('source_base should be:', root)
            break

Found Forks at: /kaggle/input/datasets/ashtonmears/kaggleutensildataset3/KaggleUtensilDataset/Forks
source_base should be: /kaggle/input/datasets/ashtonmears/kaggleutensildataset3/KaggleUtensilDataset


In [23]:
import os
import shutil
import random
from pathlib import Path

random.seed(42)

source_base = '/kaggle/input/datasets/ashtonmears/kaggleutensildataset3/KaggleUtensilDataset'
class_folders = ['Forks', 'Knives', 'Spoons']

output_dir = '/kaggle/working/utensil_cls'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

class_name_map = {'Forks': 'Fork', 'Knives': 'Knife', 'Spoons': 'Spoon'}

for folder_name in class_folders:
    cls_name = class_name_map[folder_name]
    src = os.path.join(source_base, folder_name)
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.heic'))]
    random.shuffle(images)

    split_idx = int(len(images) * 0.85)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    train_dir = f'{output_dir}/train/{cls_name}'
    val_dir = f'{output_dir}/val/{cls_name}'
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    for img_name in train_imgs:
        shutil.copy(os.path.join(src, img_name), os.path.join(train_dir, img_name))
    for img_name in val_imgs:
        shutil.copy(os.path.join(src, img_name), os.path.join(val_dir, img_name))

print('Classification dataset prepared:')
for split in ['train', 'val']:
    for cls_name in ['Fork', 'Knife', 'Spoon']:
        count = len(os.listdir(f'{output_dir}/{split}/{cls_name}'))
        print(f'  {split}/{cls_name}: {count} images')

Classification dataset prepared:
  train/Fork: 265 images
  train/Knife: 208 images
  train/Spoon: 314 images
  val/Fork: 47 images
  val/Knife: 37 images
  val/Spoon: 56 images


In [17]:
yaml_content = """
path: /kaggle/working/utensil_yolo
train: images/train
val:   images/val

nc: 3
names: ['Fork', 'Knife', 'Spoon']
"""

with open('/kaggle/working/utensil_yolo/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('dataset.yaml written.')

dataset.yaml written.


In [24]:
from ultralytics import YOLO

# YOLO has a classification model variant — yolo11n-cls.pt
# This loads the classification version of the same architecture
model = YOLO('yolo11n-cls.pt')

results = model.train(
    task='classify',
    data='/kaggle/working/utensil_cls',
    epochs=30,
    patience=8,
    imgsz=224,
    batch=32,

    # Light augmentation — appropriate for classification
    hsv_h=0.025,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.05,
    scale=0.0,
    flipud=0.2,
    fliplr=0.5,

    # Regularization
    dropout=0.1,
    weight_decay=0.0005,

    # Output
    name='utensil_cls_v1',
    project='/kaggle/working',
    device=0,
)

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/utensil_cls, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.025, hsv_s=0.6, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=utensil_cls_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/utensil_yolo/dataset.yaml, degrees=20.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.25, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.03, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/kaggle/working/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=utensil_yolo_v2, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

In [20]:
import shutil

shutil.copy('/kaggle/working/utensil_yolo_v3-2/weights/best.pt',
            '/kaggle/working/utensil_yolo_v3_best.pt')

print('Ready to download: /kaggle/working/utensil_yolo_v3_best.pt')

Ready to download: /kaggle/working/utensil_yolo_v3_best.pt


In [25]:
import shutil

shutil.copy('/kaggle/working/utensil_cls_v1/weights/best.pt',
            '/kaggle/working/utensil_cls_v1_best.pt')

print('Ready to download: /kaggle/working/utensil_cls_v1_best.pt')

Ready to download: /kaggle/working/utensil_cls_v1_best.pt
